# Florence2 Large COCO Baseline

This notebook was reorganized for the GitHub reproducibility package.
Original file: `COCO-Baseline/Florence-2_BaseModel.ipynb`.

**Security note:** hard-coded Hugging Face tokens were removed. Use interactive login or environment variables instead.


In [ ]:
# 1. Uyumsuz olan dev sürümünü kaldır
!pip uninstall -y transformers

# 2. Florence-2 ile %100 uyumlu olan KARARLI sürümü kur
!pip install transformers==4.44.2
!pip install timm flash_attn einops accelerate

print("✅ Kurulum bitti. ŞİMDİ MUTLAKA RESTART YAPIN! 👇")

In [ ]:
import torch
import transformers
from transformers import AutoProcessor, AutoModelForCausalLM
from google.colab import drive
import os

# Versiyon Kontrolü (İçimiz rahat olsun)
print(f"Versiyon Kontrolü: {transformers.__version__}")
# Beklenen: 4.44.2

if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

MODEL_ID = "microsoft/Florence-2-large"
print(f"⏳ {MODEL_ID} yükleniyor...")

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    trust_remote_code=True
).to("cuda").eval()

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)

print("✅ BAŞARDIK! Model Hatasız Yüklendi.")

In [ ]:
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')

In [ ]:
# Dosya Yolları ve Kontrol ---- Zaman aşımına uğradığında da yok diyebilir örn COCO_val2014_000000391895, driveda baktım var :)

import os
import json

# ==========================================
# AYARLAR (PALI3Gemma.ipynb ile aynı yapıda)
# ==========================================
COCO_ROOT = "/content/drive/MyDrive/datasets/coco2014"
KARPATHY_TEST = "/content/drive/MyDrive/coco_karpathy/coco_karpathy_test.json"

print(f"📂 JSON Dosyası: {KARPATHY_TEST}")
print(f"📂 Resim Kökü : {COCO_ROOT}")

# JSON Yükle
try:
    with open(KARPATHY_TEST, "r") as f:
        items = json.load(f)
    print(f"✅ JSON başarıyla okundu. Toplam kayıt: {len(items)}")
except Exception as e:
    print(f"❌ JSON okunamadı! Yol yanlış olabilir. Hata: {e}")
    items = []

# Basit Yol Kontrolü (İlk 5 resim)
print("\n🚀 Yol Testi (İlk 5 resim)...")
for i, item in enumerate(items[:5]):
    img_rel = item["image"]
    full_path = os.path.join(COCO_ROOT, img_rel)
    if os.path.exists(full_path):
        print(f"   ✅ [OK] {full_path}")
    else:
        print(f"   ❌ [YOK] {full_path}")

In [ ]:
# Tahmin - Inference

from PIL import Image
import torch
import re
import os
import json

# Çıktı Dosyası
OUTPUT_FILE = "florence2_large_preds.json"

# ID Parse Fonksiyonu
def coco_id_from_relpath(rel_path: str) -> int:
    m = re.search(r"_(\d{12})\.jpg$", rel_path)
    if not m: return int(rel_path.split('_')[-1].split('.')[0])
    return int(m.group(1))

preds = []
print(f"🚀 {len(items)} resim için Florence-2 Testi Başlıyor (Task: <CAPTION>)...")

# Florence-2 Görev Tanımı
TASK_PROMPT = "<CAPTION>"

for i, item in enumerate(items):
    try:
        # Resim Yolu
        img_rel = item["image"]
        img_path = os.path.join(COCO_ROOT, img_rel)
        image_id = coco_id_from_relpath(img_rel)

        # Resmi Yükle
        image = Image.open(img_path).convert('RGB')

        # Florence-2 Girdisi
        # model.device ve torch_dtype otomatik algılanır
        inputs = processor(text=TASK_PROMPT, images=image, return_tensors="pt").to(model.device, model.dtype)

        # Üretim (Generation)
        generated_ids = model.generate(
            input_ids=inputs["input_ids"],
            pixel_values=inputs["pixel_values"],
            max_new_tokens=1024, # Florence-2 kendi içinde duracağı yeri bilir
            do_sample=False,
            num_beams=3
        )

        # Çözme (Decoding)
        generated_text = processor.batch_decode(generated_ids, skip_special_tokens=False)[0]

        # Parse (Task sonucunu temizle)
        parsed_answer = processor.post_process_generation(
            generated_text,
            task=TASK_PROMPT,
            image_size=(image.width, image.height)
        )

        # Sadece caption metnini al
        caption = parsed_answer[TASK_PROMPT]

        preds.append({
            "image_id": image_id,
            "caption": caption
        })

    except Exception as e:
        print(f"⚠️ Hata (ID: {image_id}): {e}")
        preds.append({"image_id": image_id, "caption": "error"})

    # İlerleme (Her 100 resimde bir)
    if i % 100 == 0:
        print(f"[{i}/{len(items)}] {caption}")

# Kaydet
with open(OUTPUT_FILE, "w") as f:
    json.dump(preds, f)

print(f"\n✅ İşlem bitti! Tahminler kaydedildi: {OUTPUT_FILE}")

In [ ]:
from pycocotools.coco import COCO
from pycocoevalcap.eval import COCOEvalCap
from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.rouge.rouge import Rouge
from pycocoevalcap.cider.cider import Cider
import json

# Yollar (Senin standart yolların)
GT_PATH = "/content/drive/MyDrive/coco_karpathy/coco_karpathy_test_gt.json"
PREDS_FILE = "florence2_large_preds.json"

print("📊 Değerlendirme Başlıyor (Java Gerektirmeyen Mod)...")

try:
    # 1. Verileri Yükle
    coco = COCO(GT_PATH)
    cocoRes = coco.loadRes(PREDS_FILE)

    # 2. Evaluator Başlat
    cocoEval = COCOEvalCap(coco, cocoRes)

    # 3. ID Eşleşmesi (Hata önlemi)
    imgIds = sorted([img['image_id'] for img in json.load(open(PREDS_FILE))])
    cocoEval.params["image_id"] = imgIds

    # ====================================================
    # 🔥 KRİTİK AYAR: SADECE GÜVENLİ METRİKLERİ SEÇİYORUZ
    # ====================================================
    # Normalde kütüphane SPICE ve METEOR'u otomatik ekler ve Java hatası verir.
    # Biz burada listeyi elle vererek o sorunlu metrikleri iptal ediyoruz.
    cocoEval.scorers = [
        (Bleu(4), ["Bleu_1", "Bleu_2", "Bleu_3", "Bleu_4"]),
        (Rouge(), "ROUGE_L"),
        (Cider(), "CIDEr")
    ]

    # 4. Hesapla
    cocoEval.evaluate()

    # 5. Sonuçları Bas
    print("\n" + "="*40)
    print("🏆 FLORENCE-2 LARGE SONUÇLARI (Java'sız)")
    print("="*40)
    print(f"CIDEr    : {cocoEval.eval['CIDEr']:.3f}")
    print(f"ROUGE-L  : {cocoEval.eval['ROUGE_L']:.3f}")
    print(f"BLEU-4   : {cocoEval.eval['Bleu_4']:.3f}")
    print(f"BLEU-1   : {cocoEval.eval['Bleu_1']:.3f}")
    print("="*40)

except Exception as e:
    print(f"⚠️ Bir hata oluştu: {e}")

In [ ]:
# Drive'a Yedekleme Yapalım!!

import shutil
import os
import json

# Klasör Yapısı
SAVE_DIR = "/content/drive/MyDrive/tez_sonuclar/florence2_large"
os.makedirs(SAVE_DIR, exist_ok=True)

# 1. Tahmin Dosyasını Yedekle
SOURCE_PREDS = "florence2_large_preds.json"
TARGET_PREDS = os.path.join(SAVE_DIR, "florence2_large_preds_final.json")

if os.path.exists(SOURCE_PREDS):
    shutil.copy(SOURCE_PREDS, TARGET_PREDS)
    print(f"✅ Tahminler yedeklendi: {TARGET_PREDS}")

# 2. Skorları Kaydet
metrics = {
    "Model": "Florence-2-large",
    "Params": "0.77B",
    "Scores": {
        "CIDEr": cocoEval.eval['CIDEr'],
        "BLEU_4": cocoEval.eval['Bleu_4'],
        "ROUGE_L": cocoEval.eval['ROUGE_L'],
        "BLEU_1": cocoEval.eval['Bleu_1']
    }
}

# JSON Formatında
with open(os.path.join(SAVE_DIR, "florence2_large_metrics.json"), "w") as f:
    json.dump(metrics, f, indent=4)

# TXT Formatında (Okunabilir Rapor)
with open(os.path.join(SAVE_DIR, "florence2_large_scores.txt"), "w") as f:
    f.write(f"=== {metrics['Model']} ===\n")
    f.write(f"Parametre: {metrics['Params']}\n")
    f.write("-" * 20 + "\n")
    for k, v in metrics['Scores'].items():
        f.write(f"{k:<10}: {v:.3f}\n")

print(f"✅ Rapor kaydedildi: {SAVE_DIR}")

In [ ]:
# Model parametrelerini sayalım;

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("="*30)
print(f"🧠 MODEL: Florence-2-Large")
print("="*30)
print(f"Toplam Parametre: {total_params:,}")
print(f"Toplam (Milyar) : {total_params / 1e9:.3f} B")
print("-" * 30)
print(f"Eğitilebilir    : {trainable_params:,}")
print("="*30)